In [10]:
import pandas as pd
import numpy as np
import pickle

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [12]:
df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [13]:
df.drop("customerID", axis=1, inplace=True)

In [14]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.dropna(inplace=True)
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=[0, 1, 2, 3]
)

In [15]:
df["TenureGroup"] = df["TenureGroup"].astype(int)

In [17]:
label_encoders = {}
for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [18]:
X = df.drop("Churn",axis=1)
y = df["Churn"]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [21]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [22]:
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    scale_pos_weight=len(y_train[y_train == 0]) / len(y_train[y_train == 1]),
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

model.fit(X_train_res, y_train_res)


C:\Users\works\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [01:45:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [23]:
y_probs = model.predict_proba(X_test)[:, 1]

In [24]:
print("\n🔍 Threshold Tuning Results:\n")

thresholds = [0.5, 0.4, 0.3, 0.25, 0.2]

for t in thresholds:
    print(f"\n--- Threshold: {t} ---")
    
    y_pred = (y_probs > t).astype(int)
    
    print(classification_report(y_test, y_pred))


🔍 Threshold Tuning Results:


--- Threshold: 0.5 ---
              precision    recall  f1-score   support

           0       0.89      0.70      0.78      1033
           1       0.48      0.76      0.59       374

    accuracy                           0.71      1407
   macro avg       0.68      0.73      0.68      1407
weighted avg       0.78      0.71      0.73      1407


--- Threshold: 0.4 ---
              precision    recall  f1-score   support

           0       0.90      0.63      0.74      1033
           1       0.44      0.82      0.58       374

    accuracy                           0.68      1407
   macro avg       0.67      0.72      0.66      1407
weighted avg       0.78      0.68      0.70      1407


--- Threshold: 0.3 ---
              precision    recall  f1-score   support

           0       0.93      0.58      0.71      1033
           1       0.43      0.89      0.58       374

    accuracy                           0.66      1407
   macro avg       0.68   

In [25]:
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

print("\n✅ Model saved successfully!")


✅ Model saved successfully!
